# CNN Feature Extraction

This notebook generates the `lstm_features_*V2.npy` files used by the training notebook.
The pre-computed feature files are already included in `data/` — you only need to run
this notebook if you want to regenerate them from scratch.

**Run from the repository root directory.**

Set `DATA_DIR` in the first code cell to point to the desired seismic input folder.

In [ ]:
import os

# ── USER SETTINGS ────────────────────────────────────────────
# Set to the seismic input you want to process, relative to repo root
SEISMIC_CASE = 'BI-BNCS_ICA100'   # change to ICA50, ICA140, CNP100, SP100, LAC100
DATA_DIR = os.path.join(os.getcwd(), 'data', SEISMIC_CASE)
OUTPUT_NPY = os.path.join(DATA_DIR, f'lstm_features_{SEISMIC_CASE.split("_")[-1]}V2.npy')
# ──────────────────────────────────────────────────────────────
print(f'Data directory: {DATA_DIR}')
print(f'Output file:    {OUTPUT_NPY}')


Regular

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, InputLayer, Dropout
import random # Import random
import tensorflow as tf # Corrected: Import tensorflow as tf


seed_value = 12345
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)


# -----------------------
# Parameters
# -----------------------
Fs = 200                   # Sampling frequency in Hz
dt = 1.0 / Fs              # Sampling interval (s)
window_size = 200         # Number of samples per window (tunable)
feature_dim = 100         # Output feature vector dimension (changed from 50 to 200)

# -----------------------
# Load seismic input from file
# The file "acel_abajo_ICA140.txt" has two columns: first is time, second is acceleration.
file_path = r"G:\My Drive\FBLSTM\FB-CNP100\u.txt"
data = np.loadtxt(file_path)
time_full = data[:]     # Full time vector (s)
SP_input_full = data[:]   # Seismic acceleration

# Optionally, if you want to process a specific time frame, you can subset here.
# For now we use the entire signal.
# -----------------------
# Build spectrograms for each window
# -----------------------
n_fft = 64          # FFT window length for STFT
hop_length = 16     # Hop length

n_total = len(SP_input_full)
n_windows = n_total // window_size

spectrograms = []
for w in range(n_windows):
    start = w * window_size
    end = start + window_size
    window_signal = SP_input_full[start:end]
    S = librosa.stft(window_signal, n_fft=n_fft, hop_length=hop_length, window='hann')
    S_mag = np.abs(S)
    spectrograms.append(S_mag)
spectrograms = np.array(spectrograms)  # shape: (n_windows, freq_bins, time_frames)
print("Spectrograms shape:", spectrograms.shape)

# Add a channel dimension (assume grayscale image)
spectrograms = spectrograms[..., np.newaxis]
input_shape = spectrograms.shape[1:]  # (freq_bins, time_frames, 1)
print("CNN Input shape:", input_shape)

# -----------------------
# Plot an example spectrogram (for the first window)
# -----------------------
plt.figure(figsize=(8, 4))
librosa.display.specshow(20*np.log10(spectrograms[0,...,0] + 1e-6), sr=Fs, hop_length=hop_length,
                         x_axis='time', y_axis='linear')
plt.title("Example Spectrogram for Window 0")
plt.colorbar(label='dB')
plt.show()

# -----------------------
# Build a CNN model that outputs a feature vector of size 'feature_dim'
# -----------------------
model = Sequential([
    InputLayer(input_shape=input_shape),
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(feature_dim, activation='linear')  # Now outputs a 200-dimensional vector
])
model.summary()

# -----------------------
# (Optional) Compile the model (if training is intended)
# -----------------------
model.compile(optimizer='adam', loss='mse')

# -----------------------
# Run the model on the spectrogram dataset to extract features
# -----------------------
output_vectors = model.predict(spectrograms)
print("Output feature vectors shape:", output_vectors.shape)
# Each row in output_vectors is a 200-dimensional vector corresponding to one window.

# Plot the feature vector for the first window
plt.figure(figsize=(10,4))
plt.stem(output_vectors[10])
plt.xlabel("Feature Index")
plt.ylabel("Value")
plt.title("Extracted Feature Vector (Size 200) for Window 0")
plt.grid(True)
plt.show()

# -----------------------
# Save the feature vectors to file for later use in your LSTM network.
# -----------------------
np.save("lstm_features_CNP100_FB.npy", output_vectors)
print("Feature vectors saved to 'lstm_features.npy'.")